<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.5rem; margin-bottom: 0.2rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.4rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfire 
</h2>
  <hr class="title-rule> style=<margin-bottom: 5rem;">
</div>

## Module 3: *Feature Engineering*
##### Version Number: 4.0
---
### Contents  
> 1. *Calculate Temporal Fields*
> 2. *Calculate Lagged Weather Variables*
> 3. *Build Target*
> 4. *Fire History*
> 5. *File Export*
---
### Notes
> Features Added:
> - `Year`,`Month`,`Season` Simple temporal factors
> - `Fire_History` An average count of fires per month in a region spanning the alst two years (needs refinement)
> - `Lagged_Variables` 7 day rolling averages for key select weather factors.
---
### Inputs
- `samples_projected.csv` cleaned weather data joined with cleaned fire damage dataset
---
### Outputs 
- `engineered_samples.csv` - final clean dataset with features added
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Show basic facts about a dataframe
from src.data_utils import basic_explore

# basic health checks after a merge
from src.data_utils import post_merge_check

---
### Third Party Dependencies

In [2]:
# Core data tools
import pandas as pd
import numpy as np
from datetime import datetime

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

---

## Load Datasets

In [3]:
samples = pd.read_csv("../data/processed/samples_projected.csv")

---

## 1. Calculate Temporal Fields
`Season` `Year`

In [4]:
def get_season(date):
    month = date.month
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'
    
## Apply function   
samples['Date'] = pd.to_datetime(samples['Date'])
samples['Season'] = samples['Date'].apply(get_season)
samples['Year'] = samples['Date'].dt.year

## 2. Calculate Lagged Weather Variables

Calculate 7-day rolling averages for select weather variables to capture recent trends that may influence fire severity.

In [5]:
# Sort data by County and Date to prepare for rolling average
samples = samples.sort_values(by=['grid_id', 'Date'])

# Define columns for 7-day rolling average
avg_columns = [
    "Vapor Pressure Deficit",
    "Precipitation",
    "Solar Radiation",
    "Daily Maximum Air Temperature",
    "Daily Minimum Air Temperature",
    "Maximum Relative Humidity",
    "Minimum Relative Humidity",
    "Wind Speed"
]

# Apply rolling mean by County
for col in avg_columns:
    samples[f'{col} 7 Day Avg'] = samples.groupby('grid_id')[col].rolling(window=7, min_periods=1).mean().reset_index(level=0, drop=True)

## 4. Build Final Target

Create a categorical target that represents the risk levels of damage caused.\
\
`Low` Risk = 0, No fires present on that day in a grid\
`Medium` Risk = 1, Fires present in the grid\
`High` Risk = 2, Fires present that cause financial damage or have above average acreage burns in the grid

In [6]:
mean_damage = samples['total_fire_damage'].mean()
mean_acres = samples['acres'].mean()

def fire_risk_category(row, mean_acre, mean_dam):
    # High risk: damaging fires
    if (row['total_fire_damage'] > 0) or (row['acres'] > mean_acre):
        return 2

    # Moderate risk: at least one fire
    elif row['fire_count'] > 0:
        return 1

    # Low risk: all other cases
    else:
        return 0

samples['Target'] = samples.apply(
    fire_risk_category,
    axis=1,
    mean_acre=mean_acres,
    mean_dam=mean_damage
)

## Display resulting risk category assignments
samples['Target'].value_counts()

Target
0    476070
1    107920
2     24890
Name: count, dtype: int64

#### (TODO) Visualize Target

## 5.  Historical fires per month

Create a new column that records the historical average number of fires for each month, based on all previous years up to that point. Each row should reflect the average number of fires that occurred in the same calendar month across prior years, providing a historical baseline for comparison.

**ASSUMPTION** -Since the weather coverage in this dataset is limited, no previous year readings exists.
For now, I will just use the current count as a rough estimation for the average. 

In [7]:
# Convert Datetime column to string
samples['Month_Year'] = samples['Date'].dt.strftime('%Y-%m')
month_year = samples['Month_Year'].unique()

#### (TODO) Validate

In [8]:
# Dictionary to store month-to-count mapping
monthly_fire_counts = {}

# Step 1: Calculate fire counts for each month
for month in month_year:
    # Get index and subset
    month_index = samples[samples['Month_Year'] == month].index
    subset = samples.loc[month_index]
    
    # Eliminate 'Low' days
    total = subset[subset['Target'] != 0]
    
    # Store count
    monthly_fire_counts[month] = len(total)

# Step 2: Calculate 2-year rolling average
# Convert month_year to sorted list to ensure order
month_year_sorted = sorted(month_year)

# Dictionary for rolling averages
rolling_avg = {}

for i, month in enumerate(month_year_sorted):
    if i == 0:
        # First year: average of all *future* years (exclude current)
        future_values = list(monthly_fire_counts.values())[1:]
        rolling_avg[month] = np.mean(future_values) if future_values else 0

    elif i == 1:
        # Second year: average of first year and current
        prev = month_year_sorted[0]
        rolling_avg[month] = np.mean([monthly_fire_counts[prev], monthly_fire_counts[month]])
    
    else:
        # Rolling average of previous 2 years
        prev1 = month_year_sorted[i - 1]
        prev2 = month_year_sorted[i - 2]
        rolling_avg[month] = np.mean([monthly_fire_counts[prev1], monthly_fire_counts[prev2]])

# Step 3: Assign back to each row in the dataframe
for month in month_year_sorted:
    month_index = samples[samples['Month_Year'] == month].index
    samples.loc[month_index, '2-Year Avg Fires'] = rolling_avg[month]

In [9]:
drop = ['Month_Year']
samples = samples.drop(columns=drop)

# Santa Ana

`Wind speed x degree of dryness` An experimental score to capture the influence of the Santa Ana winds.

In [10]:
samples['Santa_Ana_Score'] = (
    samples['Wind Speed'] * (100 - samples['Minimum Relative Humidity']) / 100
)

---

## Interaction Features

#### (TODO) Expand interaction definitions

In [11]:
# Define interaction pairs
interactions = [
    ("Wind Speed", "100-hour Dead Fuel Moisture"), ## Fire spread
    ("Vapor Pressure Deficit", "Solar Radiation"), #
    ("Precipitation", "1000-hour Dead Fuel Moisture"), #Fire start potential
    ("northness_mean", "Daily Maximum Air Temperature"), 
    ("road_density", "forest_percent"), # interace between human density and forests
    ("power_line_density", "total_housing"), # Human environment interaction
    ("2-Year Avg Fires", "Precipitation 7 Day Avg"), # Capture historical patterns
]

In [12]:
interaction_columns = []

# Create the interaction columns for the main dataset
for v1, v2 in interactions:
    col_name = f"{v1}_x_{v2}"
    samples[col_name] = samples[v1] * samples[v2]
    interaction_columns.append(col_name)
    
interaction_columns.append('Target')

## Wind Slope Interactions

In [13]:
topo_cols = ['slope_mean', 'slope_max', 'northness_mean', 'eastness_mean', 'elevation_mean', 'elevation_range']
wind_cols = ['Wind Speed', 'Wind Speed 7 Day Avg']
wind_slope = []

# Create the interaction columns
for v1 in wind_cols:
    for v2 in topo_cols:
        col_name = f"{v1}_x_{v2}"
        samples[col_name] = samples[v1] * samples[v2]
        wind_slope.append(col_name)
    
wind_slope.append('Target')

## 6. Export Data

In [14]:
samples.to_csv("../data/processed/engineered_samples.csv", index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
